<a href="https://colab.research.google.com/github/esrakirbas/msc-thesis-reducing-hallucinations-in-medical-diagnosis/blob/main/experiments/Medical_Grounding_001_Data_Preparation_questions_and_corpus_from_MedQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info("awinml/medqa") #get current revision number of MedQA (afdb980b627374a7f10a0a11ed68455596c52742)
print(info.sha)

afdb980b627374a7f10a0a11ed68455596c52742


In [ ]:
from datasets import load_dataset
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(questions)
#to explore the first 20 questions (I will select some diagnosis questions among them. SELECTED indices : [2, 6, 15, 19] )
for i, item in enumerate(questions["test"].select(range(20))):
    print(f"\nQuestion {i+1} (index {i}):")
    print(item["question"])
    print(item["options"])
    print(item["answer"])
    print("-" * 60)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    validation: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1272
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

Question 1 (index 0):
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication o

In [ ]:
from datasets import load_dataset
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(books)
#to explore the text books and find the relevant one's indexes. MANUALLY SELECTED TEXT BOOKS : [8]:InternalMed_Harrison and [17]:Surgery_Schwartz
for i, item in enumerate(books["train"]):
  print(f"\nText Book {i} : {item["title"]}")


corpus/train.jsonl: reconstructing file:   0%|          |  0.00B / 89.9MB            

corpus/train.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'title', 'source_filename', 'text'],
        num_rows: 18
    })
})

Text Book 0 : Anatomy_Gray

Text Book 1 : Biochemistry_Lippincott

Text Book 2 : Cell_Biology_Alberts

Text Book 3 : First_Aid_Step1

Text Book 4 : First_Aid_Step2

Text Book 5 : Gynecology_Novak

Text Book 6 : Histology_Ross

Text Book 7 : Immunology_Janeway

Text Book 8 : InternalMed_Harrison

Text Book 9 : Neurology_Adams

Text Book 10 : Obstentrics_Williams

Text Book 11 : Pathology_Robbins

Text Book 12 : Pathoma_Husain

Text Book 13 : Pediatrics_Nelson

Text Book 14 : Pharmacology_Katzung

Text Book 15 : Physiology_Levy

Text Book 16 : Psichiatry_DSM-5

Text Book 17 : Surgery_Schwartz


In [ ]:
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17] #InternalMed_Harrison and Surgery_Schwartz
selected_textbooks = books["train"].select(selected_textbook_indices)
#to explore the structure of the books
print(selected_textbooks.column_names)
for i, book in enumerate(selected_textbooks):
  print("\nBook Title:", book["title"])
  print("Text length:", len(book["text"]))
  print("First 1000 characters:")
  print(book["text"][:1000])

['doc_id', 'title', 'source_filename', 'text']

Book Title: InternalMed_Harrison
Text length: 22312859
First 1000 characters:
xxxviii its related products in new formats. Kim Davis, as Associate Managing We are privileged to have compiled this 19th edition and areEditor, has adeptly ensured that the complex production of this multi-enthusiastic about all that it offers our readers. We learned much in the authored textbook proceeded smoothly and efficiently. Dominik Pucek process of editing Harrison’s and hope that you will find this edition a oversaw the production of the new procedural videos and Priscilla uniquely valuable educational resource. Beer expertly oversaw the production of our extensive DVD content. The Editors Jeffrey Herzich ably served as production manager for this new edition.

Chapter 1 The Practice of Medicine the practice of Medicine The Editors THE PHYSICIAN IN THE TWENTY-FIRST CENTURY No greater opportunity, responsibility, or obligation can fall to the lot of a 

In [37]:
import re #built in regular exp. lib.
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17]
selected_textbooks = books["train"].select(selected_textbook_indices)
book_InternalMed = selected_textbooks[0]
book_Surgery = selected_textbooks[1]
#try to find a related passage as a grounding context for the first question selected_index = 2
text = book_InternalMed["text"]
keyword = "sedative-hypnotic" #for question 16 : "porcelain gallbladder" for question 7 : "urinary tract obstruction" , for question 3 : "atheroembolism"
matches = list(re.finditer(re.escape(keyword), text, re.IGNORECASE)) #escape means treat special chars as normal text
#to see the matched passages. After investigating I used those positions to get the related paragraphs
#for match in matches: #commented out after finding the candidate grounding paragraph
#    position = match.start()
#    print("Position:", position)
#    print(text[max(0, position-100):position+2000])

grounding_contexts = [None] * len(selected_questions)
#Grounding Context for Question 3 - index:2
related_paragraph_start_position = text.find("Atheroembolism is another cause of limb ischemia. In this conditi")
if related_paragraph_start_position != -1:
  #related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
  related_paragraph_end_position = text.find("\n", related_paragraph_start_position)
  if related_paragraph_end_position != -1:
    grounding_contexts[0] = text[related_paragraph_start_position:related_paragraph_end_position]
print("grounding_contexts[0] : ", grounding_contexts[0][:300])
#Grounding Context for Question 7 - index:6
chapter_start = text.find("urinary tract Obstruction Julian L. Seifter") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.find("Approach to the Patient with Gastrointestinal Disease",chapter_start)
grounding_contexts[1] = text[chapter_start:chapter_end]
print("grounding_contexts[1] : ", grounding_contexts[1][:300])
#Grounding Context for Question 16 - index:15
chapter_start = text.lower().find("limey (milk of calcium) bile and porcelain gallbladder") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("carcinoma of the gallbladder.",chapter_start) + len("carcinoma of the gallbladder.")
grounding_contexts[2] = text[chapter_start:chapter_end]
print("grounding_contexts[2] : ", grounding_contexts[2][:300])
#Grounding Context for Question 20 - index:19 (2 paragraphs - founded grounding context provides partial evidence.)
chapter_start = text.lower().find("in striking contrast is the hypoactive subtype") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("with prominent apathy and psychomotor slowing.",chapter_start)+len("with prominent apathy and psychomotor slowing.")
grounding_contexts[3] = text[chapter_start:chapter_end]
chapter_start = text.lower().find("in contrast, in opioid and sedative-hypnotic poisoning") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("grade 3 or 4 physiologic depression; [table 473e-2]).",chapter_start) + len("grade 3 or 4 physiologic depression; [table 473e-2]).")
grounding_contexts[3] += "\n\n"
grounding_contexts[3] += text[chapter_start:chapter_end]
print("grounding_contexts[3] : ", grounding_contexts[3][:300])

#print the grounding data to a txt
#with open("grounding_contexts.txt", "w", encoding="utf-8") as f:
#    for i, context in enumerate(grounding_contexts):
#        f.write(f"\n{'='*60}\n")
#        f.write(f"Grounding Context {i} - Question {selected_question_indices[i] + 1}\n")
#        f.write(f"{'='*60}\n")
#        f.write(context + "\n")
# After creating this file, I manually corrected some formatting errors,
# such as removing misplaced tables and page numbers.




grounding_contexts[0] :  Atheroembolism is another cause of limb ischemia. In this condition, multiple small deposits of fibrin, platelets, and cholesterol debris embolize from proximal atherosclerotic lesions or aneurysmal sites. Large protruding aortic atheromas are a source of emboli that may lead to limb ischemia, as we
grounding_contexts[1] :  urinary tract Obstruction Julian L. Seifter Obstruction to the flow of urine, with attendant stasis and elevation in urinary tract pressure, impairs renal and urinary conduit functions and is a common cause of acute and chronic kidney disease (obstruc-tive nephropathy). With early relief of obstruct
grounding_contexts[2] :  limey (milk of calcium) bile anD porcelain GallblaDDer Calcium salts in the lumen of the gallbladder in sufficient concentration may produce calcium precipitation and diffuse, hazy opacification of bile or a layering effect on plain abdominal roentgenography. This so-called limey bile, or milk of ca
grounding_contexts[3] 